In [ ]:
"""
POC: Data Quality & Schema Validation for Kafka-Flink-MinIO Pipeline
Repo: streaming-glynac
Results output: MinIO (mocked locally when not connected)
"""

import json
import time
import random
import logging
import statistics
from datetime import datetime, timezone
from dataclasses import dataclass, field
from typing import Any
from io import StringIO

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Schema Definition
# ---------------------------------------------------------------------------
SCHEMA = {
    "event_id":    {"type": str,   "required": True,  "min_len": 1},
    "user_id":     {"type": int,   "required": True,  "min": 1},
    "event_type":  {"type": str,   "required": True,  "allowed": {"click", "view", "purchase"}},
    "timestamp":   {"type": str,   "required": True},   # ISO-8601
    "amount":      {"type": float, "required": False, "min": 0.0},
    "metadata":    {"type": dict,  "required": False},
}


# ---------------------------------------------------------------------------
# Schema Validator
# ---------------------------------------------------------------------------
class SchemaValidator:
    """Validates a Kafka message payload against SCHEMA."""

    def validate(self, record: dict) -> tuple[bool, list[str]]:
        errors: list[str] = []

        for field_name, rules in SCHEMA.items():
            value = record.get(field_name)

            # Required check
            if rules.get("required") and value is None:
                errors.append(f"Missing required field: '{field_name}'")
                continue

            if value is None:
                continue  # optional, absent — skip

            # Type check
            if not isinstance(value, rules["type"]):
                errors.append(
                    f"'{field_name}' expected {rules['type'].__name__}, "
                    f"got {type(value).__name__}"
                )
                continue

            # String constraints
            if rules["type"] is str:
                if "min_len" in rules and len(value) < rules["min_len"]:
                    errors.append(f"'{field_name}' too short (min {rules['min_len']} char)")
                if "allowed" in rules and value not in rules["allowed"]:
                    errors.append(
                        f"'{field_name}' value '{value}' not in allowed set {rules['allowed']}"
                    )

            # Numeric constraints
            if rules["type"] in (int, float):
                if "min" in rules and value < rules["min"]:
                    errors.append(f"'{field_name}' = {value} below minimum {rules['min']}")

        return (len(errors) == 0, errors)


# ---------------------------------------------------------------------------
# Data Quality Checker
# ---------------------------------------------------------------------------
class DataQualityChecker:
    """
    Checks real-world data quality issues:
      - nulls in critical fields
      - duplicates (by event_id)
      - future timestamps
      - negative amounts
      - outlier amounts (> 3 std-dev)
    """

    def __init__(self):
        self._seen_ids: set[str] = set()

    def check(self, record: dict, all_amounts: list[float]) -> tuple[bool, list[str]]:
        issues: list[str] = []

        # Null / empty critical fields
        for critical in ("event_id", "user_id", "event_type", "timestamp"):
            if not record.get(critical):
                issues.append(f"Null/empty critical field: '{critical}'")

        # Duplicate detection
        eid = record.get("event_id", "")
        if eid in self._seen_ids:
            issues.append(f"Duplicate event_id: '{eid}'")
        else:
            self._seen_ids.add(eid)

        # Future timestamp
        ts_str = record.get("timestamp", "")
        try:
            ts = datetime.fromisoformat(ts_str.replace("Z", "+00:00"))
            if ts > datetime.now(timezone.utc):
                issues.append(f"Future timestamp detected: {ts_str}")
        except (ValueError, AttributeError):
            issues.append(f"Unparseable timestamp: '{ts_str}'")

        # Negative amount
        amount = record.get("amount")
        if amount is not None and amount < 0:
            issues.append(f"Negative amount: {amount}")

        # Statistical outlier (requires at least 2 data points)
        if amount is not None and len(all_amounts) >= 2:
            mean = statistics.mean(all_amounts)
            stdev = statistics.stdev(all_amounts)
            if stdev > 0 and abs(amount - mean) > 3 * stdev:
                issues.append(
                    f"Amount {amount:.2f} is a statistical outlier "
                    f"(mean={mean:.2f}, stdev={stdev:.2f})"
                )

        return (len(issues) == 0, issues)


# ---------------------------------------------------------------------------
# Test Data Factory
# ---------------------------------------------------------------------------
def generate_records(n: int) -> list[dict]:
    """
    Produces a realistic mix of valid, invalid, and edge-case records.
    Approximately: 70% valid | 20% invalid | 10% edge cases
    """
    records = []
    for i in range(n):
        scenario = random.choices(
            ["valid", "invalid", "edge"], weights=[70, 20, 10]
        )[0]

        base = {
            "event_id":   f"evt-{i:06d}",
            "user_id":    random.randint(1, 9999),
            "event_type": random.choice(["click", "view", "purchase"]),
            "timestamp":  datetime.now(timezone.utc).isoformat(),
            "amount":     round(random.uniform(1.0, 500.0), 2),
            "metadata":   {"source": "kafka-poc"},
        }

        if scenario == "invalid":
            mutation = random.choice([
                "missing_required",
                "wrong_type",
                "bad_event_type",
                "negative_amount",
                "duplicate_id",
            ])
            if mutation == "missing_required":
                base.pop(random.choice(["event_id", "user_id", "event_type"]))
            elif mutation == "wrong_type":
                base["user_id"] = "not-an-int"
            elif mutation == "bad_event_type":
                base["event_type"] = "unknown_action"
            elif mutation == "negative_amount":
                base["amount"] = -abs(base["amount"])
            elif mutation == "duplicate_id":
                base["event_id"] = "evt-000000"  # forced duplicate

        elif scenario == "edge":
            edge = random.choice([
                "future_ts",
                "extreme_amount",
                "null_optional",
                "empty_metadata",
            ])
            if edge == "future_ts":
                base["timestamp"] = "2099-01-01T00:00:00+00:00"
            elif edge == "extreme_amount":
                base["amount"] = 999_999.99
            elif edge == "null_optional":
                base["amount"] = None
            elif edge == "empty_metadata":
                base["metadata"] = {}

        records.append(base)
    return records


# ---------------------------------------------------------------------------
# Performance Benchmark
# ---------------------------------------------------------------------------
@dataclass
class BenchmarkResult:
    volume: int
    elapsed_ms: float
    throughput_rps: float
    schema_pass: int
    schema_fail: int
    quality_pass: int
    quality_fail: int


def run_benchmark(volume: int) -> BenchmarkResult:
    """Validates `volume` records and returns timing + quality metrics."""
    validator = SchemaValidator()
    checker   = DataQualityChecker()
    records   = generate_records(volume)
    amounts   = [r["amount"] for r in records if isinstance(r.get("amount"), (int, float))]

    s_pass = s_fail = q_pass = q_fail = 0

    start = time.perf_counter()
    for rec in records:
        ok_s, _ = validator.validate(rec)
        ok_q, _ = checker.check(rec, amounts)
        s_pass += ok_s;  s_fail += not ok_s
        q_pass += ok_q;  q_fail += not ok_q
    elapsed_ms = (time.perf_counter() - start) * 1000

    return BenchmarkResult(
        volume=volume,
        elapsed_ms=elapsed_ms,
        throughput_rps=volume / (elapsed_ms / 1000),
        schema_pass=s_pass, schema_fail=s_fail,
        quality_pass=q_pass, quality_fail=q_fail,
    )


# ---------------------------------------------------------------------------
# MinIO Output (mock when offline)
# ---------------------------------------------------------------------------
class MinIOClient:
    """
    Thin wrapper — writes to MinIO when connected,
    falls back to local JSON file when offline.
    """

    def __init__(self, bucket: str = "poc-results", offline: bool = True):
        self.bucket  = bucket
        self.offline = offline

    def put(self, key: str, data: dict) -> str:
        payload = json.dumps(data, indent=2, default=str)
        if self.offline:
            path = f"/tmp/{key.replace('/', '_')}"
            with open(path, "w") as fh:
                fh.write(payload)
            log.info("[MinIO-mock] Saved → %s", path)
            return path
        # Real MinIO call would go here (minio-py SDK)
        raise NotImplementedError("Live MinIO not configured in this POC run")


# ---------------------------------------------------------------------------
# Report Builder
# ---------------------------------------------------------------------------
def build_report(benchmarks: list[BenchmarkResult], scenario_results: list[dict]) -> dict:
    return {
        "poc_meta": {
            "run_at":    datetime.now(timezone.utc).isoformat(),
            "repo":      "streaming-glynac",
            "pipeline":  "Kafka → Flink → MinIO",
        },
        "scenario_results": scenario_results,
        "performance_benchmarks": [
            {
                "volume":           b.volume,
                "elapsed_ms":       round(b.elapsed_ms, 2),
                "throughput_rps":   round(b.throughput_rps, 0),
                "schema": {"pass": b.schema_pass, "fail": b.schema_fail},
                "quality": {"pass": b.quality_pass, "fail": b.quality_fail},
            }
            for b in benchmarks
        ],
        "production_readiness": {
            "throughput_adequate":  all(b.throughput_rps > 5_000 for b in benchmarks),
            "schema_coverage":      "Full field + type + constraint validation",
            "quality_checks":       ["null", "duplicate", "future_ts", "negative", "outlier"],
            "deployment_notes": [
                "Replace MinIOClient.offline=True with live endpoint + credentials",
                "Inject Kafka consumer (confluent-kafka) ahead of validate()/check()",
                "Wire Flink via PyFlink or REST API for streaming window support",
                "Add Prometheus metrics endpoint for throughput / error-rate alerting",
            ],
        },
    }


# ---------------------------------------------------------------------------
# Scenario Suite
# ---------------------------------------------------------------------------
def run_scenario_suite() -> list[dict]:
    """
    Exercises the validator and checker against hand-crafted scenarios
    covering valid records, known-bad records, and edge cases.
    """
    validator = SchemaValidator()
    checker   = DataQualityChecker()

    test_cases = [
        # (label, record, expected_schema_ok, expected_quality_ok)
        ("Valid purchase", {
            "event_id": "evt-valid-001", "user_id": 42,
            "event_type": "purchase", "amount": 29.99,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "metadata": {"promo": "none"},
        }, True, True),

        ("Missing event_id", {
            "user_id": 7, "event_type": "click",
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }, False, False),

        ("Wrong type for user_id", {
            "event_id": "evt-003", "user_id": "abc",
            "event_type": "view",
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }, False, True),

        ("Invalid event_type", {
            "event_id": "evt-004", "user_id": 5,
            "event_type": "DELETE",
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }, False, True),

        ("Negative amount", {
            "event_id": "evt-005", "user_id": 9,
            "event_type": "purchase", "amount": -10.0,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }, False, False),

        ("Future timestamp", {
            "event_id": "evt-006", "user_id": 11,
            "event_type": "click",
            "timestamp": "2099-12-31T00:00:00+00:00",
        }, True, False),

        ("Duplicate event_id", {
            "event_id": "evt-valid-001",   # same as first test
            "user_id": 99, "event_type": "view",
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }, True, False),
    ]

    results = []
    amounts = [29.99, 15.0, 50.0]  # small baseline for outlier detection

    for label, record, exp_schema, exp_quality in test_cases:
        ok_s, errs_s = validator.validate(record)
        ok_q, errs_q = checker.check(record, amounts)
        passed = (ok_s == exp_schema) and (ok_q == exp_quality)

        results.append({
            "scenario":      label,
            "schema_ok":     ok_s,
            "quality_ok":    ok_q,
            "expectation_met": passed,
            "schema_errors": errs_s,
            "quality_issues": errs_q,
        })

        status = "✓ PASS" if passed else "✗ FAIL"
        log.info("[Scenario] %-35s → %s", label, status)

    return results


# ---------------------------------------------------------------------------
# Entry Point
# ---------------------------------------------------------------------------
def main():
    log.info("=== POC: Data Quality & Schema Validation ===")

    # Task 1 & 2: Scenario suite (hand-crafted + known-bad records)
    log.info("--- Running scenario suite ---")
    scenario_results = run_scenario_suite()

    # Task 3: Performance benchmarks at different volumes
    log.info("--- Running performance benchmarks ---")
    volumes = [100, 1_000, 10_000, 50_000]
    benchmarks = []
    for vol in volumes:
        result = run_benchmark(vol)
        benchmarks.append(result)
        log.info(
            "  Volume %6d → %.1f ms | %.0f rec/s | "
            "schema fail=%d | quality fail=%d",
            vol, result.elapsed_ms, result.throughput_rps,
            result.schema_fail, result.quality_fail,
        )

    # Build & persist report
    report = build_report(benchmarks, scenario_results)
    minio  = MinIOClient(offline=True)          # set offline=False for live MinIO
    path   = minio.put("poc-results/run_latest.json", report)

    log.info("=== POC Complete. Report → %s ===", path)

    # Quick pass/fail summary to stdout
    total   = len(scenario_results)
    passed  = sum(1 for r in scenario_results if r["expectation_met"])
    log.info("Scenario suite: %d/%d expectations met", passed, total)

    prod_ok = report["production_readiness"]["throughput_adequate"]
    log.info("Production readiness (throughput): %s", "READY" if prod_ok else "NEEDS REVIEW")


if __name__ == "__main__":
    main()

2026-05-06 11:23:06 [INFO] === POC: Data Quality & Schema Validation ===
2026-05-06 11:23:06 [INFO] --- Running scenario suite ---
2026-05-06 11:23:06 [INFO] [Scenario] Valid purchase                      → ✓ PASS
2026-05-06 11:23:06 [INFO] [Scenario] Missing event_id                    → ✓ PASS
2026-05-06 11:23:06 [INFO] [Scenario] Wrong type for user_id              → ✓ PASS
2026-05-06 11:23:06 [INFO] [Scenario] Invalid event_type                  → ✓ PASS
2026-05-06 11:23:06 [INFO] [Scenario] Negative amount                     → ✓ PASS
2026-05-06 11:23:06 [INFO] [Scenario] Future timestamp                    → ✓ PASS
2026-05-06 11:23:06 [INFO] [Scenario] Duplicate event_id                  → ✓ PASS
2026-05-06 11:23:06 [INFO] --- Running performance benchmarks ---
2026-05-06 11:23:06 [INFO]   Volume    100 → 30.2 ms | 3309 rec/s | schema fail=20 | quality fail=15
2026-05-06 11:23:08 [INFO]   Volume   1000 → 1466.4 ms | 682 rec/s | schema fail=161 | quality fail=165
